In [1]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [2]:
import os
os.chdir("/content/gdrive/My Drive/TL_A/DeepKale/experiments_0.2/")

In [3]:
%%capture
!pip install optuna torch plotly botorch

In [4]:
# Import Packages
import os
import pandas as pd
import datetime
import time
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import optuna
import matplotlib.pyplot as plt
import pickle as pkl

In [5]:
from src.preprocessing.data_preparation import DataPreparation
from src.preprocessing.feature_scaling import robust_scaling, min_max_scaling
from src.preprocessing.feature_engineering import feature_encoding, create_temporal_features, create_lag_features, expanding_mean_std_weighted_avg
from src.utils.helper_functions import get_approach

In [6]:
warnings.filterwarnings("ignore")

# Set seed for numpy
SEED = 1

# # Set seed for PyTorch
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

Device: cpu



In [8]:
if os.path.exists('/mnt/work/dkale/dkale_Colab/experiments_0.2/'):
    # Define base directory path
    root_dir = '/mnt/work/dkale/dkale_Colab/experiments_0.2/'
else:
    root_dir = '/content/gdrive/My Drive/TL_A/DeepKale/experiments_0.2/'

current_date = datetime.datetime.now().strftime("%d%m%Y")
approach = get_approach(model_type=0)

In [9]:
N_FUTURE = 1
N_PAST = 7

In [10]:
# Define subdirectories
root_data_dir = root_dir + 'data/'
preprocessed_dir = root_data_dir + 'preprocessed/ACN/'

In [11]:
base_df = pd.read_csv(rf'{preprocessed_dir}acn_caltech_jpl_0.2_0.1.22_06_2023.csv')

In [12]:
base_df = feature_encoding(base_df)

In [13]:
base_df['siteID_19'] = 0

In [14]:
base_df = create_temporal_features(base_df)

In [15]:
base_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16548 entries, 0 to 16547
Data columns (total 30 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   connectionTime                16548 non-null  datetime64[ns]
 1   kWhDelivered_Smoothed         16548 non-null  float64       
 2   total_charging_time_Smoothed  16548 non-null  float64       
 3   idle_time_Smoothed            16548 non-null  float64       
 4   siteID_0                      16548 non-null  int64         
 5   siteID_1                      16548 non-null  int64         
 6   siteID_1_2                    16548 non-null  int64         
 7   siteID_2                      16548 non-null  int64         
 8   siteID_19                     16548 non-null  int64         
 9   Hour_of_Day                   16548 non-null  int64         
 10  Day_Of_Week                   16548 non-null  int64         
 11  Day_Of_year                 

In [16]:
base_df = create_lag_features(dataframe = base_df, target = base_df['kWhDelivered_Smoothed'], thres=0.15)

In [17]:
base_df = expanding_mean_std_weighted_avg(dataframe = base_df, window_size = 2)

In [18]:
base_robust_cols = ['kWhDelivered_Smoothed', 'total_charging_time_Smoothed', 'idle_time_Smoothed', 'lag_1', 'lag_2', 'expanding_mean', 'expanding_std', 'weighted_avg']
base_sin_cos_cols = ['Hour_of_Day', 'Day_Of_Week', 'Day_Of_year', 'Month_Of_Year']
base_scaled_df, base_robust_scaling_params = robust_scaling(df = base_df, robust_cols = base_robust_cols, sin_cos_cols = base_sin_cos_cols)

In [19]:
base_scaled_df

,connectionTime,kWhDelivered_Smoothed,total_charging_time_Smoothed,idle_time_Smoothed,siteID_0,siteID_1,siteID_1_2,siteID_2,siteID_19,Time_of_day_0_4,...,expanding_std,weighted_avg,Hour_of_Day_sin,Hour_of_Day_cos,Day_Of_Week_sin,Day_Of_Week_cos,Day_Of_year_sin,Day_Of_year_cos,Month_Of_Year_sin,Month_Of_Year_cos
2,2018-04-25 08:00:00,0.746376,0.646602,2.553822,0,0,0,1,0,0,...,-8.294165,0.583808,0.816970,-0.576680,0.866025,-0.5,0.917584,-0.397543,0.866025,-5.000000e-01
3,2018-04-25 09:00:00,0.146241,0.658738,2.796656,0,0,0,1,0,0,...,-6.004664,0.519849,0.631088,-0.775711,0.866025,-0.5,0.917584,-0.397543,0.866025,-5.000000e-01
4,2018-04-25 10:00:00,0.788525,1.537670,1.665897,0,0,0,1,0,0,...,-5.938415,0.324112,0.398401,-0.917211,0.866025,-0.5,0.917584,-0.397543,0.866025,-5.000000e-01
5,2018-04-25 11:00:00,0.169348,0.617937,3.621904,0,0,0,1,0,0,...,-5.560825,0.557526,0.136167,-0.990686,0.866025,-0.5,0.917584,-0.397543,0.866025,-5.000000e-01
6,2018-04-25 12:00:00,-0.217433,-0.121801,-0.169116,0,0,0,1,0,0,...,-3.961496,-0.012560,-0.136167,-0.990686,0.866025,-0.5,0.917584,-0.397543,0.866025,-5.000000e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16543,2020-03-14 13:00:00,-0.364984,-0.435793,-0.150901,1,0,0,0,0,0,...,0.324533,-0.303580,-0.398401,-0.917211,-0.866025,0.5,0.956235,0.292600,1.000000,6.123234e-17
16544,2020-03-14 14:00:00,-0.461636,-0.491063,-0.160549,0,1,0,0,0,0,...,0.324490,-0.473105,-0.631088,-0.775711,-0.866025,0.5,0.956235,0.292600,1.000000,6.123234e-17
16545,2020-03-14 15:00:00,0.389824,-0.015246,-0.169162,0,0,0,1,0,0,...,0.324236,-0.242228,-0.816970,-0.576680,-0.866025,0.5,0.956235,0.292600,1.000000,6.123234e-17
16546,2020-03-14 16:00:00,-0.077461,0.038388,-0.004618,0,0,0,1,0,0,...,0.323941,0.191224,-0.942261,-0.334880,-0.866025,0.5,0.956235,0.292600,1.000000,6.123234e-17


In [20]:
base_robust_scaling_params

{'kWhDelivered_Smoothed': (3.890024258872975, 6.967769930949368),
 'total_charging_time_Smoothed': (1.05776681437644, 1.9361205079300299),
 'idle_time_Smoothed': (0.30617052685750346, 1.7989144460820348),
 'lag_1': (3.8908053645380516, 6.9688548871565485),
 'lag_2': (3.8920108644147904, 6.970298117609429),
 'expanding_mean': (5.036894038880169, 0.353426811024546),
 'expanding_std': (4.121420716122482, 0.3955828822528069),
 'weighted_avg': (2.5528964788322748, 3.972580711020572)}